# SQL Agent with Prebuilt HITL & PII Redaction Middleware in LangChain

This notebook demonstrates how to implement **Human-in-the-loop (HITL) Approval** and **PII (Personally Identifiable Information) Redaction** using **prebuilt LangChain agent middlewares**.

Instead of writing custom logic, we import and configure the native middlewares from `langchain.agents.middleware`:
1. **`HumanInTheLoopMiddleware`**: Pauses graph execution and triggers an `interrupt` when specific tools are called. The execution can then be approved, rejected, or edited and resumed using a `Command(resume=...)` payload.
2. **`PIIMiddleware`**: Automatically scans tool inputs, tool outputs, or model messages using regex and Luhn algorithm validation (for credit cards) to redact sensitive PII (emails, credit card numbers, etc.).

## 1. Setup & Environment Variables
Set up the API keys and initialize the virtual environment variables.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load keys from root directory .env if present
load_dotenv(dotenv_path="../.env")

if "GEMINI_API_KEY" not in os.environ:
    if "GOOGLE_API_KEY" in os.environ:
        os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]
    else:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI API Key: ")

## 2. Initialize SQL Database & Define Query Tool
We'll create a local in-memory SQLite database containing sensitive columns (names, emails, credit card numbers, and balances) and wrap a query function as a LangChain `@tool`.

*Note: The mock credit card numbers are valid Visa and Mastercard test cards that pass the Luhn algorithm validation required by the prebuilt PII middleware.*

In [2]:
import sqlite3
from langchain.tools import tool

# 1. Create SQLite database in-memory and insert sensitive records
db_conn = sqlite3.connect(":memory:", check_same_thread=False)
cursor = db_conn.cursor()

cursor.execute("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    credit_card TEXT,
    balance REAL
)
""")

users_data = [
    (1, "Alice Smith", "alice.smith@example.com", "4111-1111-1111-1111", 1250.00), # Valid Visa test card
    (2, "Bob Jones", "bob.jones@gmail.com", "5555-5555-5555-4444", 3500.50)      # Valid Mastercard test card
]

cursor.executemany("INSERT INTO users VALUES (?, ?, ?, ?, ?)", users_data)
db_conn.commit()

# 2. Define standard SELECT-only query tool
@tool("query_sql_db", description="Executes a read-only SQL SELECT query against the users database and returns the result as text. Input must be a valid SQL SELECT query.")
def query_sql_db(query: str) -> str:
    """Query the SQL database."""
    try:
        # Verify query is a SELECT query (read-only)
        clean_query = query.strip().upper()
        if not clean_query.startswith("SELECT"):
            return "Error: Only read-only SELECT queries are allowed."
        
        conn_cursor = db_conn.cursor()
        conn_cursor.execute(query)
        rows = conn_cursor.fetchall()
        
        if not rows:
            return "Query executed successfully. No rows returned."
        
        # Format output as a nice string table
        columns = [description[0] for description in conn_cursor.description]
        output = " | ".join(columns) + "\n" + "-"*50 + "\n"
        for row in rows:
            output += " | ".join(str(val) for val in row) + "\n"
        return output
    except Exception as e:
        return f"SQL Query execution failed with error: {str(e)}"

## 3. Instantiate Prebuilt Middlewares
We import the native middlewares from `langchain.agents.middleware`:
- `HumanInTheLoopMiddleware`: Set up to trigger an interrupt when `query_sql_db` is called.
- `PIIMiddleware`: Set up to redact both `credit_card` and `email` content in tool execution results.

In [3]:
from langchain.agents.middleware.human_in_the_loop import HumanInTheLoopMiddleware
from langchain.agents.middleware.pii import PIIMiddleware

# 1. HITL Approval Middleware (interrupts query_sql_db execution)
approval_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "query_sql_db": {
            "allowed_decisions": ["approve", "reject"]
        }
    }
)

# 2. PII Redaction Middlewares for credit card and email
pii_cc_middleware = PIIMiddleware(
    pii_type="credit_card",
    strategy="redact",
    apply_to_tool_results=True
)

pii_email_middleware = PIIMiddleware(
    pii_type="email",
    strategy="redact",
    apply_to_tool_results=True
)

print("Prebuilt middlewares successfully initialized!")

Prebuilt middlewares successfully initialized!


## 4. Instantiate the SQL Agent with Checkpointer
We configure the agent with `MemorySaver` checkpointing, allowing state preservation when an interrupt halts the graph.

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

# Initialize Gemini Chat Model
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# State persistence checkpointer
memory = MemorySaver()

# Compile Agent
sql_agent = create_agent(
    model=llm,
    tools=[query_sql_db],
    system_prompt=(
        "You are a database assistant. Your goal is to run query_sql_db to retrieve user "
        "account details. Synthesize and report the query results. "
        "Always respect redactions (e.g. [REDACTED_CREDIT_CARD]) in your final response."
    ),
    middleware=[approval_middleware, pii_cc_middleware, pii_email_middleware],
    checkpointer=memory
)
print("SQL Agent with prebuilt middlewares and checkpointer is ready!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


SQL Agent with prebuilt middlewares and checkpointer is ready!


## 5. Demonstration Runs
We'll execute three scenarios demonstrating how the agent handles interrupts, PII redactions, and standard allowed data access.

### Scenario 1: Approved Query with PII Redaction
We query Alice's sensitive data. The agent will halt on an interrupt. We will inspect the interrupt, approve it, and see the PII redacted in the tool results.

In [5]:
from langgraph.types import Command

# Config with a thread_id for state tracking
config_1 = {"configurable": {"thread_id": "run_thread_1"}}
query_1 = "Can you fetch Alice Smith's credit card and email address?"

print(f"User Request: {query_1}\n")

# 1. Start execution (expecting an interrupt to halt the run)
result = sql_agent.invoke({"messages": [{"role": "user", "content": query_1}]}, config=config_1)

# 2. Inspect graph state and verify it is paused on an interrupt
state = sql_agent.get_state(config_1)
print(f"-> Graph execution state: {state.next}")

if state.next:
    for task in state.tasks:
        if task.interrupts:
            print("\n🛑 [HITL Middleware Triggered]")
            print(f"   Action Requests: {task.interrupts[0].value.get('action_requests')}")
            print(f"   Allowed Decisions: {task.interrupts[0].value.get('review_configs')}\n")

# 3. Resume execution with an APPROVED decision
print("--> Sending APPROVAL command to resume execution...")
resume_decision = {"decisions": [{"type": "approve"}]}
final_result = sql_agent.invoke(Command(resume=resume_decision), config=config_1)

print("\n=== FINAL AGENT RESPONSE ===\n")
print(final_result["messages"][-1].content)

User Request: Can you fetch Alice Smith's credit card and email address?

-> Graph execution state: ('HumanInTheLoopMiddleware.after_model',)

🛑 [HITL Middleware Triggered]
   Action Requests: [{'name': 'query_sql_db', 'args': {'query': "SELECT email, credit_card FROM users WHERE name = 'Alice Smith'"}, 'description': 'Tool execution requires approval\n\nTool: query_sql_db\nArgs: {\'query\': "SELECT email, credit_card FROM users WHERE name = \'Alice Smith\'"}'}]
   Allowed Decisions: [{'action_name': 'query_sql_db', 'allowed_decisions': ['approve', 'reject']}]

--> Sending APPROVAL command to resume execution...

=== FINAL AGENT RESPONSE ===

[{'type': 'text', 'text': "Alice Smith's email address and credit card information are both redacted.", 'extras': {'signature': 'EjQKMgEMOdbHIwWv6sCOLTyc+oFegWwvFvz8eWfPDzOeUjhOY58qdepThs5AQjbh+eVn+ze0'}}]


### Scenario 2: Rejected Query
We query Bob's balance, but reject the tool execution when prompted. The agent will gracefully handle the rejection.

In [6]:
config_2 = {"configurable": {"thread_id": "run_thread_2"}}
query_2 = "What is Bob Jones's balance?"

print(f"User Request: {query_2}\n")

# 1. Start execution
result = sql_agent.invoke({"messages": [{"role": "user", "content": query_2}]}, config=config_2)

# 2. Check for interrupt
state = sql_agent.get_state(config_2)
if state.next:
    print("\n🛑 [HITL Middleware Triggered]")
    print(f"   Action Requests: {state.tasks[0].interrupts[0].value.get('action_requests')}\n")

# 3. Resume execution with a REJECTED decision
print("--> Sending REJECTION command to resume execution...")
resume_decision = {"decisions": [{"type": "reject", "message": "Access to financial balance denied by human reviewer."}]}
final_result = sql_agent.invoke(Command(resume=resume_decision), config=config_2)

print("\n=== FINAL AGENT RESPONSE ===\n")
print(final_result["messages"][-1].content)

User Request: What is Bob Jones's balance?


🛑 [HITL Middleware Triggered]
   Action Requests: [{'name': 'query_sql_db', 'args': {'query': "SELECT balance FROM users WHERE name = 'Bob Jones'"}, 'description': 'Tool execution requires approval\n\nTool: query_sql_db\nArgs: {\'query\': "SELECT balance FROM users WHERE name = \'Bob Jones\'"}'}]

--> Sending REJECTION command to resume execution...

=== FINAL AGENT RESPONSE ===

[{'type': 'text', 'text': 'I am sorry, but I am unable to retrieve the balance for Bob Jones as access to that information has been restricted.', 'extras': {'signature': 'EjQKMgEMOdbHgvOcjuN68/3H0J7GAsATdEwrDVG2qmuN6pK2h6FZxwuGKWlU4ZlUsAzDgUza'}}]


### Scenario 3: Approved Query without PII (Normal Run)
We ask for Alice Smith's account balance. We approve the query, and verify that the balance is returned without any redactions since it doesn't contain PII.

In [7]:
config_3 = {"configurable": {"thread_id": "run_thread_3"}}
query_3 = "What is Alice Smith's balance?"

print(f"User Request: {query_3}\n")

# 1. Start execution
result = sql_agent.invoke({"messages": [{"role": "user", "content": query_3}]}, config=config_3)

# 2. Check for interrupt
state = sql_agent.get_state(config_3)
if state.next:
    print("\n🛑 [HITL Middleware Triggered]")
    print(f"   Action Requests: {state.tasks[0].interrupts[0].value.get('action_requests')}\n")

# 3. Resume execution with an APPROVED decision
print("--> Sending APPROVAL command to resume execution...")
resume_decision = {"decisions": [{"type": "approve"}]}
final_result = sql_agent.invoke(Command(resume=resume_decision), config=config_3)

print("\n=== FINAL AGENT RESPONSE ===\n")
print(final_result["messages"][-1].content)

User Request: What is Alice Smith's balance?


🛑 [HITL Middleware Triggered]
   Action Requests: [{'name': 'query_sql_db', 'args': {'query': "SELECT balance FROM users WHERE name = 'Alice Smith'"}, 'description': 'Tool execution requires approval\n\nTool: query_sql_db\nArgs: {\'query\': "SELECT balance FROM users WHERE name = \'Alice Smith\'"}'}]

--> Sending APPROVAL command to resume execution...

=== FINAL AGENT RESPONSE ===

[{'type': 'text', 'text': "Alice Smith's balance is 1250.0.", 'extras': {'signature': 'EjQKMgEMOdbH3f+E9LnIG7y1IVKSD6xinVxSXQ/egziGudR6SXbwRPFdF9N1re+63w8rraIR'}}]
